# MANHWA STUDIO — Google Colab + Google Drive

Interface Gradio com tema preto e vermelho, prévia do vídeo e moldura, links do Google Drive, renderização FFmpeg e exportação MP4 para o Drive.

**GPU recomendada: NVIDIA T4.** O notebook verifica NVENC e usa CPU automaticamente se o encoder de GPU falhar. Não baixa vídeos do YouTube.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/ManhwaStudio')
(BASE/'entrada').mkdir(parents=True, exist_ok=True)
(BASE/'exportados').mkdir(parents=True, exist_ok=True)
print('Google Drive pronto:', BASE)


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade gradio google-api-python-client google-auth pillow
import subprocess, shutil
print(subprocess.run(['ffmpeg','-version'],capture_output=True,text=True).stdout.splitlines()[0])
gpu = subprocess.run(['nvidia-smi'],capture_output=True,text=True)
print(gpu.stdout[:1500] if gpu.returncode == 0 else 'GPU não detectada: selecione T4 em Ambiente de execução > Alterar tipo de ambiente de execução > GPU.')
enc = subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True).stdout
print('NVENC disponível no FFmpeg:', 'h264_nvenc' in enc)


In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio
!python -m py_compile colab/renderer.py colab/app_gradio.py
print('Arquivos verificados.')


## Iniciar o estúdio

Execute a célula abaixo. Aguarde aparecer o link público `gradio.live` e abra-o. Cole os links compartilháveis dos arquivos no Drive. Para links privados, o Colab solicitará autenticação Google quando acessar o Drive.

O MP4 final é salvo automaticamente em `Meu Drive/ManhwaStudio/exportados`.

In [ ]:
# Autentique o Google Drive no contexto principal do Colab (não dentro do Gradio)
from google.colab import auth
auth.authenticate_user()
import google.auth
from pathlib import Path
credentials, project = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
credential_path = Path('/content/colab_drive_credentials.json')
credential_path.write_text(credentials.to_json())
print('Google Drive autenticado. Credenciais temporárias preparadas para o painel.')


In [ ]:
%cd /content/manhwa-recreator-studio
!python colab/app_gradio.py
